##Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

####Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max, count, avg
from pyspark.sql import Window
import pyspark.sql.functions as F

### Geolocation Bronze Table Data Manipulation and Cleaning

In [0]:
df_geo_bronze = spark.table("olist_ecommerce_project.bronze.brz_geolocation")

In [0]:
# Basic profiling
print("Total rows:", df_geo_bronze.count())
print("Distinct zip prefixes:", df_geo_bronze.select("geolocation_zip_code_prefix").distinct().count())


In [0]:
# Null check
df_geo_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_geo_bronze.columns
]).show()

In [0]:
# Check lat/lng ranges
# Brazil's actual bounding box:
# Latitude:  -33.75 to  5.27 (negative = south)
# Longitude: -73.99 to -34.79 (all negative for Brazil)
df_geo_bronze.select(
    spark_min("geolocation_lat").alias("min_lat"),
    spark_max("geolocation_lat").alias("max_lat"),
    spark_min("geolocation_lng").alias("min_lng"),
    spark_max("geolocation_lng").alias("max_lng")
).show()

##### Checking Outliers in Latitude and Longitude as seeing which and how many values are outside brasil

In [0]:
# Brazil bounding box
LAT_MIN, LAT_MAX = -33.75, 5.27
LNG_MIN, LNG_MAX = -73.99, -34.79

# How many rows fall OUTSIDE Brazil's bounding box?
outliers = df_geo_bronze.filter(
    (col("geolocation_lat") < LAT_MIN) |
    (col("geolocation_lat") > LAT_MAX) |
    (col("geolocation_lng") < LNG_MIN) |
    (col("geolocation_lng") > LNG_MAX)
)

print("Outlier rows (outside Brazil):", outliers.count())
print("Valid rows (inside Brazil):", df_geo_bronze.count() - outliers.count())
print("Outlier percentage:", round(outliers.count() / df_geo_bronze.count() * 100, 2), "%")

# Also peek at some outlier examples
outliers.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state"
).show(10, truncate=False)

Filter out outliers outside Brazil's bounding box

In [0]:
# Brazil bounding box constants
LAT_MIN, LAT_MAX = -33.75, 5.27
LNG_MIN, LNG_MAX = -73.99, -34.79

# Keep only rows within Brazil's boundaries
df_geo_valid = df_geo_bronze.filter(
    (col("geolocation_lat") >= LAT_MIN) &
    (col("geolocation_lat") <= LAT_MAX) &
    (col("geolocation_lng") >= LNG_MIN) &
    (col("geolocation_lng") <= LNG_MAX)
)

print("Rows before filtering:", df_geo_bronze.count())
print("Rows after filtering:", df_geo_valid.count())
print("Rows dropped:", df_geo_bronze.count() - df_geo_valid.count())

Get most frequent city name per zip prefix

In [0]:

# Count how many times each city/state combo appears per zip prefix
df_city_counts = (
    df_geo_valid
    .groupBy(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    )
    .agg(count("*").alias("city_count"))
)

# Sanity check — see what it looks like
df_city_counts.orderBy("geolocation_zip_code_prefix").show(20, truncate=False)

In [0]:
# Use row_number() instead of rank() to handle ties
window_city = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("city_count"))

df_city_winner = (
    df_city_counts
    .withColumn("row_num", F.row_number().over(window_city))
    .filter(col("row_num") == 1)
    .drop("city_count", "row_num")
)

# Sanity check
print("Distinct zip prefixes after picking winner:", df_city_winner.count())
df_city_winner.orderBy("geolocation_zip_code_prefix").show(10, truncate=False)

Average lat/lng per zip prefix

In [0]:

df_geo_avg = (
    df_geo_valid
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("geolocation_lat"),
        avg("geolocation_lng").alias("geolocation_lng")
    )
)

# Sanity check
print("Rows after averaging:", df_geo_avg.count())
df_geo_avg.orderBy("geolocation_zip_code_prefix").show(10, truncate=False)

Join averaged coordinates with city winners

In [0]:
df_geo_silver = df_geo_avg.join(
    df_city_winner,
    on="geolocation_zip_code_prefix",
    how="left"
)

# Add ingestion timestamp, drop source file
df_geo_silver = df_geo_silver.withColumn("_ingestion_timestamp", F.current_timestamp())

# Sanity check
print("Final silver geolocation rows:", df_geo_silver.count())
df_geo_silver.orderBy("geolocation_zip_code_prefix").show(10, truncate=False)

##### Creating the Geolocation Silver Table

In [0]:
(
    df_geo_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_geolocation")
)

print("slv_geolocation written successfully")